# Домашка 1 — Послание с TRAPPIST-1d

## Подгружаем данные

In [ ]:
import pandas as pd
import numpy as np
import hashlib
from collections import Counter

key = pd.read_csv('key.csv')
message = pd.read_csv('message.csv')
codebook = pd.read_csv('codebook.csv')

N = len(codebook)  # размер алфавита
print(key.shape, message.shape, codebook.shape)
print('Размер алфавита N =', N)
key.head()

## 1. Выключенная станция

In [ ]:
missing_per_relay = key.groupby('relay')['shift'].apply(lambda s: s.isna().sum())
missing_per_relay.sort_values(ascending=False)

In [ ]:
station_off = missing_per_relay.idxmax()
print('Задача 1. Выключенная станция:', station_off)

## 2. Станция с одним числом

In [ ]:
nunique_per_relay = key.groupby('relay')['shift'].apply(lambda s: s.dropna().nunique())
nunique_per_relay.sort_values()

In [ ]:
candidates = nunique_per_relay.drop(station_off)
station_one_number = candidates.idxmin()
one_number_value = key.loc[key.relay == station_one_number, 'shift'].dropna().unique()[0]
print('Задача 2. Станция с одним числом:', station_one_number, '- число', one_number_value)

## 3. Станция с числами по кругу


In [ ]:
candidates2 = candidates.drop(station_one_number)
candidates2.sort_values().head()

In [ ]:
station_cycle = candidates2.idxmin()
cycle_values = key.loc[key.relay == station_cycle, 'shift'].dropna().unique()
print('Задача 3. Станция с числами по кругу:', station_cycle)
print('Уникальные значения:', sorted(cycle_values))
print('Задача 3б. Сколько разных чисел она передаёт:', len(cycle_values))

## 4. Станция с мусором

In [ ]:
def count_non_numeric(s):
    numeric = pd.to_numeric(s, errors='coerce')
    return ((numeric.isna()) & (s.notna())).sum()

garbage_per_relay = key.groupby('relay')['shift'].apply(count_non_numeric)
garbage_per_relay.sort_values(ascending=False)

In [ ]:
station_garbage = garbage_per_relay.idxmax()
garbage_count = garbage_per_relay[station_garbage]
print('Задача 4а. Станция с мусором:', station_garbage)
print('Задача 4б. Сколько нечисловых значений она передала:', garbage_count)

## 5. Две самые коварные станции

In [ ]:
already_found = [station_off, station_one_number, station_cycle, station_garbage]
rest_relays = [r for r in key['relay'].unique() if r not in already_found]
print(len(rest_relays), 'станций осталось разобрать:', rest_relays)

In [ ]:
rest = key[key['relay'].isin(rest_relays)].copy()
rest['shift_num'] = pd.to_numeric(rest['shift'], errors='coerce')

pivot = rest.pivot(index='pos', columns='relay', values='shift_num')
pivot.head()

In [ ]:
disagree = {r: 0 for r in rest_relays}
valid = {r: 0 for r in rest_relays}

for pos in pivot.index:
    row = pivot.loc[pos].dropna()
    if len(row) == 0:
        continue
    majority_value = Counter(row.values).most_common(1)[0][0]
    for relay, value in row.items():
        valid[relay] += 1
        if value != majority_value:
            disagree[relay] += 1

disagree_rate = pd.Series({r: disagree[r] / valid[r] for r in rest_relays}).sort_values(ascending=False)
disagree_rate

In [ ]:
sneaky_stations = disagree_rate.head(2).index.tolist()
print('Задача 5. Две самые коварные станции:', sneaky_stations)

## 6. Восстанавливаем ключ


In [ ]:
dishonest = [station_off, station_one_number, station_cycle, station_garbage] + sneaky_stations
honest_relays = [r for r in key['relay'].unique() if r not in dishonest]
print(len(honest_relays), 'честных станций:', honest_relays)

In [ ]:
honest = key[key['relay'].isin(honest_relays)].copy()
honest['shift_num'] = pd.to_numeric(honest['shift'], errors='coerce')
honest_pivot = honest.pivot(index='pos', columns='relay', values='shift_num')

final_key = []
for pos in range(honest_pivot.index.max() + 1):
    row = honest_pivot.loc[pos].dropna()
    majority_value = Counter(row.values).most_common(1)[0][0]
    final_key.append(int(majority_value))

final_key = np.array(final_key)
print('Длина ключа:', len(final_key))
print('Сумма всех сдвигов ключа:', final_key.sum())

In [ ]:

key_string = ','.join(map(str, final_key))
my_hash = hashlib.sha256(key_string.encode()).hexdigest()

with open('key_sha256.txt') as f:
    expected_hash = f.read().strip()

print('Мой хеш:      ', my_hash)
print('Хеш из файла: ', expected_hash)
print('Совпадает:', my_hash == expected_hash)

Хеш совпал — значит, ключ собран правильно.

## 7. Расшифровываем письмо


In [ ]:
code_to_char = dict(zip(codebook['code'], codebook['char']))

message_sorted = message.sort_values('pos').reset_index(drop=True)

decoded_chars = []
for pos, code in zip(message_sorted['pos'], message_sorted['code']):
    shift = final_key[pos]
    plain_code = (code - shift) % N
    decoded_chars.append(code_to_char[plain_code])

letter = ''.join(decoded_chars)
print(letter)

In [ ]:
landing_code = letter.split('площадки:')[-1].strip()
print('Задача 7а. Код посадочной площадки:', landing_code)
print('Без дефисов:', landing_code.replace('-', ''))

## Ответы

1. Выключенная станция — см. `station_off`
2. Станция с одним числом — `station_one_number`, число `one_number_value`
3. Станция с числами по кругу — `station_cycle`, число разных значений — `len(cycle_values)`
4. Станция с мусором — `station_garbage`, нечисловых значений — `garbage_count`
5. Две коварные станции — `sneaky_stations`
6. Сумма всех сдвигов ключа — `final_key.sum()`
7. Код посадочной площадки и текст письма — выше

In [ ]:
print('1. Выключенная станция:', station_off)
print('2. Станция с одним числом:', station_one_number, '| число:', one_number_value)
print('3. Станция с числами по кругу:', station_cycle, '| разных чисел:', len(cycle_values))
print('4. Станция с мусором:', station_garbage, '| нечисловых значений:', garbage_count)
print('5. Коварные станции:', sneaky_stations)
print('6. Сумма всех сдвигов ключа:', final_key.sum())
print('7. Код посадочной площадки:', landing_code)
print('7. Текст письма:', letter)